In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [3]:
!pip install -q datasets transformers sentence-transformers accelerate

In [4]:
import torch
import numpy as np

from datasets import load_dataset

from transformers import (
    AutoTokenizer,
    AutoModel,
    pipeline
)

from sentence_transformers import (
    SentenceTransformer,
    util
)

In [5]:
from datasets import load_dataset

data = load_dataset(
    "csv",
    data_files={
        "train": "/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv",
        "test": "/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv"
    }
)

train_dataset = data["train"]
test_dataset = data["test"]

print(train_dataset)
print(test_dataset)

Generating train split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

Dataset({
    features: ['id', 'prompt', 'A', 'B', 'C', 'D', 'E', 'answer'],
    num_rows: 2000
})
Dataset({
    features: ['id', 'prompt', 'A', 'B', 'C', 'D', 'E', 'answer'],
    num_rows: 500
})


In [6]:
# ============================================================
# Question 1
# ============================================================

def create_combined_text(example):
    example["combined_text"] = example["prompt"] + " " + example["A"]
    return example

train_dataset = train_dataset.map(create_combined_text)

print("Combined Text:\n")
print(train_dataset[51]["combined_text"])

length = len(train_dataset[51]["combined_text"])

print("\nCharacter Length =", length)

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Combined Text:

Determine the correct option: What is the reason behind the designation of Class L dwarfs, and what is their color and composition? among the listed options. Class L dwarfs are hotter than M stars and are designated L because L is the remaining letter alphabetically closest to M. They are bright blue in color and are brightest in ultraviolet. Their atmosphere is hot enough to allow metal hydrides and alkali metals to be prominent in their spectra. Some of these objects have masses large enough to support hydrogen fusion and are therefore stars, but most are of substellar mass and are therefore brown dwarfs.

Character Length = 614


In [7]:
# ============================================================
# Question 2
# Initialize BERT Tokenizer
# ============================================================

from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

print("Vocabulary Size:", tokenizer.vocab_size)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Vocabulary Size: 30522


In [8]:
# ============================================================
# Question 3
# [SEP] Token ID
# ============================================================

print("SEP Token:", tokenizer.sep_token)
print("SEP Token ID:", tokenizer.sep_token_id)

SEP Token: [SEP]
SEP Token ID: 102


In [11]:
# ============================================================
# Question 4
# Tokenize the entire prompt column
# ============================================================

# Convert the HF Column to a Python list
prompts = train_dataset["prompt"][:]

# (Optional) Ensure everything is a string
prompts = [str(p) for p in prompts]

tokenized_prompts = tokenizer(
    prompts,
    padding="max_length",
    truncation=True,
    max_length=128,
    return_tensors="pt"
)

print("Input IDs Shape:", tokenized_prompts["input_ids"].shape)

Input IDs Shape: torch.Size([2000, 128])


In [12]:
# ============================================================
# Question 5
# Attention Head Dimension
# ============================================================

hidden_size = 768
num_attention_heads = 12

head_dimension = hidden_size // num_attention_heads

print("Dimension of each attention head:", head_dimension)

Dimension of each attention head: 64


In [13]:
# ============================================================
# Question 6
# Load BERT and obtain last_hidden_state
# ============================================================

from transformers import AutoModel

model = AutoModel.from_pretrained("bert-base-uncased")

prompt = train_dataset[0]["prompt"]

inputs = tokenizer(
    prompt,
    return_tensors="pt"
)

with torch.no_grad():
    outputs = model(**inputs)

last_hidden_state = outputs.last_hidden_state

print("Last Hidden State Shape:", last_hidden_state.shape)

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Last Hidden State Shape: torch.Size([1, 31, 768])


In [14]:
# ============================================================
# Question 7
# CLS Embedding
# ============================================================

cls_embedding = last_hidden_state[0, 0]

first_five_sum = cls_embedding[:5].sum().item()

print("Sum of first five values:", round(first_five_sum, 4))

Sum of first five values: -1.2001


In [15]:
# ============================================================
# Question 8
# Attention Mechanism
# ============================================================

from transformers import AutoTokenizer, AutoModel
import torch

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

# Load BERT with attentions
model = AutoModel.from_pretrained(
    "bert-base-uncased",
    output_attentions=True
)

text = "Light-ion fusion is a technique."

# Tokenize
inputs = tokenizer(
    text,
    return_tensors="pt"
)

# Run model
with torch.no_grad():
    outputs = model(**inputs)

# Show tokens
tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])

print("Tokens:")
for i, token in enumerate(tokens):
    print(i, token)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Tokens:
0 [CLS]
1 light
2 -
3 ion
4 fusion
5 is
6 a
7 technique
8 .
9 [SEP]


In [16]:
# ============================================================
# Extract Attention Weight
# ============================================================

attention = outputs.attentions[-1]      # Last layer
attention_head = attention[0, 0]         # Batch 0, Head 0

fusion_index = 4  

weight = attention_head[0, fusion_index].item()

print("Attention Weight:", round(weight, 4))

Attention Weight: 0.1025


In [17]:
!pip install -q sentence-transformers scikit-learn

In [18]:
from sentence_transformers import SentenceTransformer, util
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np

In [19]:
# ============================================================
# MAP@3 Function
# ============================================================

def map_at_3(actual, predicted):

    score = 0

    for a, p in zip(actual, predicted):

        ap = 0

        for i, pred in enumerate(p[:3]):

            if pred == a:
                ap = 1/(i+1)
                break

        score += ap

    return score / len(actual)

In [20]:
# ============================================================
# TF-IDF Ranking
# ============================================================

tfidf_predictions = []

for row in train_dataset:

    prompt = row["prompt"]

    options = [
        row["A"],
        row["B"],
        row["C"],
        row["D"],
        row["E"]
    ]

    labels = ["A","B","C","D","E"]

    corpus = [prompt] + options

    vectorizer = TfidfVectorizer()

    vectors = vectorizer.fit_transform(corpus)

    prompt_vec = vectors[0]

    option_vecs = vectors[1:]

    similarities = (option_vecs @ prompt_vec.T).toarray().flatten()

    ranking = np.argsort(similarities)[::-1]

    tfidf_predictions.append([labels[i] for i in ranking[:3]])

In [21]:
# ============================================================
# MiniLM Embedding Model
# ============================================================

model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

minilm_predictions = []

for row in train_dataset:

    prompt = row["prompt"]

    options = [
        row["A"],
        row["B"],
        row["C"],
        row["D"],
        row["E"]
    ]

    labels = ["A","B","C","D","E"]

    prompt_embedding = model.encode(
        prompt,
        convert_to_tensor=True
    )

    option_embeddings = model.encode(
        options,
        convert_to_tensor=True
    )

    similarities = util.cos_sim(
        prompt_embedding,
        option_embeddings
    )[0]

    ranking = torch.argsort(
        similarities,
        descending=True
    )

    minilm_predictions.append(
        [labels[i] for i in ranking[:3]]
    )

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [22]:
answers = list(train_dataset["answer"])

mini_score = map_at_3(
    answers,
    minilm_predictions
)

print("MiniLM MAP@3 =", round(mini_score,6))

MiniLM MAP@3 = 0.423083


In [23]:
count = 0

for ans, tfidf, mini in zip(
    answers,
    tfidf_predictions,
    minilm_predictions
):

    if ans not in tfidf and ans in mini:

        count += 1

print("Improvement Count =", count)

Improvement Count = 574


In [24]:
# ============================================================
# Question 9(a)
# Prompt vs Option B Similarity
# ============================================================

from sentence_transformers import SentenceTransformer, util

model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

prompt = train_dataset[0]["prompt"]
option_b = train_dataset[0]["B"]

prompt_embedding = model.encode(prompt, convert_to_tensor=True)
option_embedding = model.encode(option_b, convert_to_tensor=True)

similarity = util.cos_sim(prompt_embedding, option_embedding)

print("Cosine Similarity =", round(similarity.item(), 4))

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Cosine Similarity = 0.7658


In [25]:
# ============================================================
# Question 10
# Zero-Shot Classification
# ============================================================

from transformers import pipeline

classifier = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli"
)

prompt = train_dataset[1]["prompt"]

candidate_labels = [
    train_dataset[1]["A"],
    train_dataset[1]["B"],
    train_dataset[1]["C"]
]

result = classifier(
    prompt,
    candidate_labels=candidate_labels
)

print(result)

print("\nTop Ranked Option:")
print(result["labels"][0])

print("\nTop Probability:")
print(round(result["scores"][0], 4))

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

{'sequence': 'What is accelerator-based light-ion fusion?', 'labels': ['Accelerator-based light-ion fusion is a technique that uses particle accelerators to achieve particle kinetic energies sufficient to induce light-ion fusion reactions. This method is relatively easy to implement and can be done in an efficient manner, requiring only a vacuum tube, a pair of electrodes, and a high-voltage transformer. Fusion can be observed with as little as 10 kV between the electrodes.', 'Accelerator-based light-ion fusion is a technique that uses particle accelerators to achieve particle kinetic energies sufficient to induce light-ion fusion reactions. This method is relatively difficult to implement and requires a complex system of vacuum tubes, electrodes, and transformers. Fusion can be observed with as little as 100 kV between the electrodes.', 'Accelerator-based light-ion fusion is a technique that uses particle accelerators to achieve particle kinetic energies sufficient to induce heavy-ion

In [26]:
# ============================================================
# Question 11
# Multi-label Zero-Shot
# ============================================================

result_multi = classifier(
    prompt,
    candidate_labels=candidate_labels,
    multi_label=True
)

print(result_multi)

softmax_sum = sum(result["scores"])
sigmoid_sum = sum(result_multi["scores"])

difference = abs(softmax_sum - sigmoid_sum)

print("\nSoftmax Sum :", softmax_sum)
print("Sigmoid Sum :", sigmoid_sum)
print("Absolute Difference :", difference)

{'sequence': 'What is accelerator-based light-ion fusion?', 'labels': ['Accelerator-based light-ion fusion is a technique that uses particle accelerators to achieve particle kinetic energies sufficient to induce light-ion fusion reactions. This method is relatively easy to implement and can be done in an efficient manner, requiring only a vacuum tube, a pair of electrodes, and a high-voltage transformer. Fusion can be observed with as little as 10 kV between the electrodes.', 'Accelerator-based light-ion fusion is a technique that uses particle accelerators to achieve particle kinetic energies sufficient to induce heavy-ion fusion reactions. This method is relatively difficult to implement and requires a complex system of vacuum tubes, electrodes, and transformers. Fusion can be observed with as little as 10 kV between the electrodes.', 'Accelerator-based light-ion fusion is a technique that uses particle accelerators to achieve particle kinetic energies sufficient to induce light-ion 

In [30]:
# ============================================================
# Question 12
# FLAN-T5 Small
# ============================================================

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# Load model and tokenizer
tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-small")
model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-small")

# Get row 0
prompt = train_dataset[0]["prompt"]
option_a = train_dataset[0]["A"]
option_b = train_dataset[0]["B"]

# Construct the exact query from the milestone
query = (
    f"Question: {prompt}. "
    f"Is the correct answer A: {option_a} or B: {option_b}? "
    f"Answer with just the letter A or B."
)

print("Prompt:\n")
print(query)

# Tokenize
inputs = tokenizer(query, return_tensors="pt")

# Generate
outputs = model.generate(
    **inputs,
    max_new_tokens=5
)

# Decode
answer = tokenizer.decode(
    outputs[0],
    skip_special_tokens=True
)

print("\nGenerated Output:")
print(answer)

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Prompt:

Question: Pick the best possible answer: What is Martin Heidegger's view on the relationship between time and human existence? among the listed options.. Is the correct answer A: Martin Heidegger believes that humans exist within a time continuum that is infinite and does not have a defined beginning or end. The relationship to the past involves acknowledging it as a historical era, and the relationship to the future involves creating a world that will endure beyond one's own time. or B: Martin Heidegger believes that humans do not exist inside time, but that they are time. The relationship to the past is a present awareness of having been, and the relationship to the future involves anticipating a potential possibility, task, or engagement.? Answer with just the letter A or B.

Generated Output:
B
